# 01 — Data preparation

Cleans the raw Google Play CSVs and writes analysis-ready parquet.

**Inputs** (must exist before running):
- `data/raw/googleplaystore.csv`
- `data/raw/googleplaystore_user_reviews.csv`

**Outputs**:
- `data/processed/apps_clean.parquet`  — 1 row per app, cleaned + engineered features
- `data/processed/reviews_clean.parquet` — 1 row per review with VADER sentiment
- `data/processed/reviews_agg.parquet` — 1 row per app, sentiment roll-ups

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
IMAGES = ROOT / "images"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
IMAGES.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import numpy as np

from src.cleaning import clean_apps_frame
from src.features import add_features
from src.nlp import vader_scores

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## Load the raw apps CSV

In [3]:
apps_raw = pd.read_csv(DATA_RAW / "googleplaystore.csv")
print(f"Raw shape: {apps_raw.shape}")
apps_raw.head()

Raw shape: (10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Clean the apps frame

Applies `parse_installs`, `parse_size_mb`, `parse_price_usd`, `parse_last_updated`, and category standardization (see `src/cleaning.py`).

In [4]:
apps = clean_apps_frame(apps_raw)
print(f"After cleaning: {apps.shape[0]:,} rows (dropped duplicates + malformed)")
apps[["App", "category", "Rating", "reviews", "installs", "size_mb", "price_usd", "last_updated"]].head()

After cleaning: 9,659 rows (dropped duplicates + malformed)


,App,category,Rating,reviews,installs,size_mb,price_usd,last_updated
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,10000.0,19.0,0.0,2018-01-07
1,Coloring book moana,ART_AND_DESIGN,3.9,967,500000.0,14.0,0.0,2018-01-15
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,5000000.0,8.7,0.0,2018-08-01
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,50000000.0,25.0,0.0,2018-06-08
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,100000.0,2.8,0.0,2018-06-20


## Add engineered features

`is_paid`, `price_band`, `log_installs`, `log_size_mb`, `age_days`, `reviews_per_install`.

In [5]:
apps = add_features(apps)
apps[["App", "category", "is_paid", "price_band", "log_installs", "age_days", "reviews_per_install"]].head()

,App,category,is_paid,price_band,log_installs,age_days,reviews_per_install
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,0,Free,9.210440,213,0.015900
1,Coloring book moana,ART_AND_DESIGN,0,Free,13.122365,205,0.001934
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,0,Free,15.424949,7,0.017502
3,Sketch - Draw & Paint,ART_AND_DESIGN,0,Free,17.727534,61,0.004313
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,0,Free,11.512935,49,0.009670


## Load and score the user-review corpus

The companion CSV ships with pre-computed `Sentiment_Polarity` and `Sentiment_Subjectivity` for a subset of rows. We re-score the full corpus with VADER so we have a consistent pipeline we control.

In [6]:
reviews_raw = pd.read_csv(DATA_RAW / "googleplaystore_user_reviews.csv")
reviews_raw = reviews_raw.dropna(subset=["Translated_Review"]).reset_index(drop=True)
print(f"Review corpus: {len(reviews_raw):,} rows covering {reviews_raw['App'].nunique():,} apps")
reviews_raw.head()

Review corpus: 37,427 rows covering 865 apps


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
3,10 Best Foods for You,Best idea us,Positive,1.00,0.300000
4,10 Best Foods for You,Best way,Positive,1.00,0.300000


In [7]:
vader = vader_scores(reviews_raw["Translated_Review"])
reviews = pd.concat([reviews_raw[["App", "Translated_Review", "Sentiment"]], vader], axis=1)
reviews = reviews.rename(columns={"Translated_Review": "review_text", "compound": "vader_compound"})
reviews.head()

,App,review_text,Sentiment,neg,neu,pos,vader_compound
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,0.0,0.466,0.534,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.0,0.481,0.519,0.6597
2,10 Best Foods for You,Works great especially going grocery store,Positive,0.0,0.549,0.451,0.6249
3,10 Best Foods for You,Best idea us,Positive,0.0,0.323,0.677,0.6369
4,10 Best Foods for You,Best way,Positive,0.0,0.192,0.808,0.6369


## Roll up per-app sentiment

Mean compound, % negative, and review count — attached back to the apps frame for modelling.

In [8]:
reviews_agg = (
    reviews.groupby("App")
    .agg(
        n_reviews_scored=("vader_compound", "size"),
        mean_compound=("vader_compound", "mean"),
        pct_negative=("vader_compound", lambda s: (s < -0.3).mean()),
        pct_positive=("vader_compound", lambda s: (s > 0.3).mean()),
    )
    .reset_index()
)
reviews_agg.describe()

,n_reviews_scored,mean_compound,pct_negative,pct_positive
count,865.000000,865.000000,865.000000,865.000000
mean,43.268208,0.337262,0.144807,0.615760
std,37.111846,0.217540,0.131628,0.197738
min,1.000000,-0.659700,0.000000,0.000000
25%,27.000000,0.211659,0.054054,0.500000
50%,36.000000,0.352476,0.114286,0.625000
75%,40.000000,0.483941,0.216216,0.750000
max,312.000000,0.981100,1.000000,1.000000


In [9]:
apps_enriched = apps.merge(reviews_agg, how="left", left_on="App", right_on="App")
print(f"Apps with review sentiment: {apps_enriched['mean_compound'].notna().sum():,} / {len(apps_enriched):,}")

Apps with review sentiment: 816 / 9,659


## Persist to parquet

In [10]:
apps_enriched.to_parquet(DATA_PROCESSED / "apps_clean.parquet", index=False)
reviews.to_parquet(DATA_PROCESSED / "reviews_clean.parquet", index=False)
reviews_agg.to_parquet(DATA_PROCESSED / "reviews_agg.parquet", index=False)
print("Written:")
for p in sorted(DATA_PROCESSED.glob("*.parquet")):
    print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size / 1024:,.0f} KB)")

Written:
  data/processed/apps_clean.parquet  (618 KB)
  data/processed/reviews_agg.parquet  (37 KB)
  data/processed/reviews_clean.parquet  (2,428 KB)
